# 05 — Deep Learning Model Zoo: 3D HemiNet, TriPlanar & Swin Transformer

**Focus:** PyTorch Model Architectures across the 7-Arm Competition Ensemble  
**Architectures Covered:**
1. **`HemiNet3D`:** Bilateral Siamese 3D ResNet sharing weights across mirrored hemispheres.
2. **`TriPlanarNet`:** 2.5D Late-Fusion Network combining Axial, Coronal, and Sagittal views.
3. **`ConvNeXt-Tiny` & `EfficientNetV2-S`:** Pretrained 2D Transfer Learning backbones.
4. **`Swin Transformer`:** Shifted Window Multi-Head Self-Attention.
5. **Self-Supervised Pretraining:** VICReg on 1,362 Unlabeled Scans.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

print(f"PyTorch Version: {torch.__version__} | CUDA Available: {torch.cuda.is_available()}")

# Architecture 1: HemiNet3D (Bilateral Siamese 3D CNN)
class HemiNet3D(nn.Module):
    """Dual-stream 3D CNN comparing left and right striatal volumes."""
    def __init__(self, in_channels=1, base_channels=24):
        super().__init__()
        # Shared 3D Encoder for Left and Right hemispheres
        self.encoder = nn.Sequential(
            nn.Conv3d(in_channels, base_channels, kernel_size=3, padding=1),
            nn.GroupNorm(4, base_channels),
            nn.GELU(),
            nn.MaxPool3d(2),
            nn.Conv3d(base_channels, base_channels*2, kernel_size=3, padding=1),
            nn.GroupNorm(4, base_channels*2),
            nn.GELU(),
            nn.AdaptiveAvgPool3d((1, 1, 1))
        )
        # Difference + Concatenation classification head
        self.head = nn.Sequential(
            nn.Linear(base_channels*4, 32),
            nn.GELU(),
            nn.Dropout(0.2),
            nn.Linear(32, 1)
        )
        
    def forward(self, left_hemi, right_hemi):
        f_left = torch.flatten(self.encoder(left_hemi), 1)
        f_right = torch.flatten(self.encoder(right_hemi), 1)
        
        # Bilateral feature interaction: [f_diff, f_max]
        diff = torch.abs(f_left - f_right)
        mean = (f_left + f_right) / 2.0
        features = torch.cat([diff, mean], dim=1)
        return self.head(features).squeeze(-1)

model_hemi = HemiNet3D()
n_params = sum(p.numel() for p in model_hemi.parameters())
print(f"HemiNet3D instantiated successfully! Parameters: {n_params:,}")

# Dummy test forward pass
x_l = torch.randn(2, 1, 24, 48, 32)
x_r = torch.randn(2, 1, 24, 48, 32)
logits = model_hemi(x_l, x_r)
print(f"Forward pass output shape: {logits.shape} (Logits: {logits.detach().numpy()})")

In [ ]:
# Architecture 2: Tri-Planar 2.5D Late Fusion Network
class TriPlanarNet(nn.Module):
    """Multi-view late fusion across axial, coronal, and sagittal projections."""
    def __init__(self, num_classes=1):
        super().__init__()
        # Shared 2D backbone for orthogonal slabs
        self.backbone = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.AdaptiveAvgPool2d((1, 1))
        )
        # View Attention / Fusion Head
        self.classifier = nn.Sequential(
            nn.Linear(64 * 3, 64),
            nn.ReLU(),
            nn.Linear(64, num_classes)
        )
        
    def forward(self, axial_slab, coronal_slab, sagittal_slab):
        f_ax = torch.flatten(self.backbone(axial_slab), 1)
        f_cor = torch.flatten(self.backbone(coronal_slab), 1)
        f_sag = torch.flatten(self.backbone(sagittal_slab), 1)
        
        f_fused = torch.cat([f_ax, f_cor, f_sag], dim=1)
        return self.classifier(f_fused).squeeze(-1)

model_tri = TriPlanarNet()
print(f"TriPlanarNet Parameters: {sum(p.numel() for p in model_tri.parameters()):,}")